# Notebook 01 - Road graph download

This notebook downloads a directed road graph for Chapinero from OpenStreetMap through OSMnx, enriches its edges with travel-time attributes, and writes the local GraphML file used by later notebooks.

Assumptions used here:
- The graph is taken from OpenStreetMap data available at execution time.
- Edge speeds are inferred by OSMnx when explicit speed data are missing.
- Capacities are assigned later from road-class heuristics.
- The resulting graph is an input artifact, not a calibrated traffic data set.

Limitations:
- OpenStreetMap changes over time, so repeated downloads may differ.
- The graph does not include observed traffic counts, O-D demand, or signal plans.


In [ ]:
import osmnx as ox
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

PLACE = 'Chapinero, Bogota, Colombia'
NETWORK_TYPE = 'drive'

ox.settings.timeout = 120
ox.settings.log_console = True

print(f'Downloading road graph for: {PLACE}')
G = ox.graph_from_place(PLACE, network_type=NETWORK_TYPE)
print(f'Nodes : {G.number_of_nodes():,}')
print(f'Edges: {G.number_of_edges():,}')

In [ ]:
nodes, edges = ox.graph_to_gdfs(G)
display(nodes.head())
display(edges[['name','length','highway','oneway','maxspeed']].head())

In [ ]:
G = ox.add_edge_speeds(G)
G = ox.add_edge_travel_times(G)
print('Speeds added.')

In [ ]:
CAPACITY_MAP = {'motorway':2200,'trunk':1800,'primary':1500,'secondary':1200,'tertiary':900,'residential':600,'unclassified':600,'living_street':300,'service':300}

def get_capacity(highway):
    if isinstance(highway, list): highway = highway[0]
    return CAPACITY_MAP.get(highway, 600)

for u, v, k, data in G.edges(data=True, keys=True):
    data['capacity'] = get_capacity(data.get('highway','residential'))
    data['t0'] = data.get('travel_time', 60)
print('Capacities added.')

In [ ]:
fig, ax = ox.plot_graph(G, node_size=5, edge_linewidth=0.5, show=False, close=False)
plt.title('Chapinero road network')
plt.savefig('../data/raw/chapinero_road_network.png', dpi=150)
plt.show()

In [ ]:
ox.save_graphml(G, '../data/raw/chapinero_drive_enriched.graphml')
print('Graph saved.')